# 04 · SFT Training: Qwen2.5-0.5B → function calling (QLoRA)

**Goal:** fine-tune on `data/sft/` and score the result **exactly** like the baseline (`03_baseline_eval`): same frozen benchmark, same prompt (`src/prompts.py`), same generator (`src/generation.py`) and same scorer (`src/evaluate.py`).

### How to run (one GPU run per day, so it has to work first time)
1. **`SMOKE_TEST = True` → Run All (~5–10 min).** A dry run of *every* cell: 10 steps on the **longest** examples (worst-case VRAM), save, merge, generate, score, save. It prints peak VRAM and an estimate of the full run time.
2. **`SMOKE_TEST = False` → Run All** and leave it. Smoke outputs go to separate `*_smoke` folders, so they never mix with real results.

### What's fixed compared with `Training_SFT_cleaned`
| Problem | Fix |
|---|---|
| `MAX_SEQ_LENGTH = 512` silently dropped 46% of the training data | read `1536` from `sft_data_report.json` (the data-prep decision, 99.5% coverage) + assert that **nothing** is dropped |
| inference prompt had no system prompt (Qwen inserts its own default) | eval uses `src.prompts.build_messages`, the same as the baseline and the training data |
| best checkpoint could be deleted | `load_best_model_at_end` on eval loss |
| a crash or sleeping laptop = lost day | resumes automatically from the last checkpoint |
| 4-bit `numel()` parameter counts were confusing | explained below and `print_trainable_parameters()` used instead |

## 0. Environment
This is a quick check that the training environment works. **`unsloth` is imported first** because it patches `transformers`/`trl`, and the patches only apply cleanly if nothing has imported those yet.

In [ ]:
import unsloth                                   # must come before transformers / trl / peft
import gc
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import peft
import torch
import transformers
import trl

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Unsloth:", unsloth.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

# find the repo root (folder containing src/) and the data folder, wherever this notebook lives
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
DATA_DIR = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "splits").is_dir())
sys.path.insert(0, str(REPO_ROOT))

from src.data_utils import read_jsonl, verify_hashes                # noqa: E402
from src.prompts import PROMPT_VERSION, build_messages              # noqa: E402

print("\nRepo root:", REPO_ROOT)
print("Data dir: ", DATA_DIR)

## 1. Config
- **`SMOKE_TEST`**: the only line you change between the dry run and the real run.
- **`NUM_EPOCHS = 1`**: 48k examples of very templated data. One epoch is usually enough, and it fits the daily GPU window. If val loss is still falling at the end, try 2 epochs in a new run.
- **Effective batch = `BATCH_SIZE × GRAD_ACCUM` = 16.** Batch 2 keeps the longest 1536-token examples within 6 GB.
- **`MAX_SEQ_LENGTH` is not typed here.** It's read from the data-prep report, so training can never disagree with the data again. The file hashes are checked too.

In [ ]:
SMOKE_TEST = True          # True = 5-10 min dry run of everything. False = the real run.

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
RUN_NAME = "sft_qwen2.5-0.5b"
BENCHMARK = "test_v1"      # must match the baseline run
SEED = 3407

# LoRA
LORA_R = 16
LORA_ALPHA = 32

# training
NUM_EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 8             # effective batch = 16
LEARNING_RATE = 2e-4
EVAL_EVERY = 200           # steps between eval-loss checks (= checkpoint interval)
SMOKE_STEPS = 10

# --- from the data-prep stage: never retype these ---
sft_report = json.loads((DATA_DIR / "sft" / "sft_data_report.json").read_text())
MAX_SEQ_LENGTH = sft_report["max_seq_len"]
RESPONSE_MARKER = sft_report["response_marker"]          # "<|im_start|>assistant\n"
assert sft_report["prompt_version"] == PROMPT_VERSION, "SFT data was built with a different prompt version"
verify_hashes(DATA_DIR / "sft", sft_report["files"])     # train/val.jsonl unchanged since data prep

# --- paths ---
if SMOKE_TEST:
    RUN_NAME += "_smoke"
OUTPUT_DIR = DATA_DIR.parent / "outputs" / RUN_NAME       # checkpoints (gitignored)
ADAPTER_DIR = OUTPUT_DIR / "adapter"                      # final LoRA adapter (small)
MERGED_DIR = OUTPUT_DIR / "merged"                        # base + adapter in bf16, used for eval and DPO
EVAL_DIR = DATA_DIR / "eval"
RESULTS_DIR = EVAL_DIR / "results" / RUN_NAME
BASE_RESULTS_DIR = EVAL_DIR / "results" / "base_qwen2.5-0.5b"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"{'SMOKE TEST' if SMOKE_TEST else 'REAL RUN'} -> {RUN_NAME}")
print("Model:", MODEL_NAME)
print(f"Max sequence length: {MAX_SEQ_LENGTH} (from data prep)")
print("Prompt version:", PROMPT_VERSION, "| data hashes ✅")
print("Seed:", SEED)

# Part A · Train

## 2. Load the base model
We use the **Instruct** checkpoint because the task is instruction → structured tool call. Unsloth returns the model-matched tokenizer, and that's the one SFT uses.

Things to check in the output: the chat template exists, EOS is `<|im_end|>` (generation stops there), and PAD is **not** EOS. If they were the same, padding positions could be confused with "stop" positions.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,                      # auto -> bf16 on the RTX 4050
)

print("Model loaded.")
print("Chat template available:", tokenizer.chat_template is not None)
print("EOS:", tokenizer.eos_token, "| ID:", tokenizer.eos_token_id)
print("PAD:", tokenizer.pad_token, "| ID:", tokenizer.pad_token_id)

## 3. Parameter inspection — total vs trainable BEFORE QLoRA
Before LoRA, the model contains **all base-model parameters**. With QLoRA those stay frozen and small trainable LoRA matrices are added on top.

- **Total parameters** = size of the original model
- **Trainable parameters** = what SFT actually updates
- **Trainable %** = how much of the model we're learning

**The 4-bit counting trap:** `bitsandbytes` packs **two** 4-bit weights into one byte, so `p.numel()` counts every quantized matrix at **half** its real size. That's why the earlier notebook showed 342M, 351M *and* 502M. The cell below prints both the raw count and the corrected one (packed tensors × 2), so you can see the difference.

In [ ]:
def count_params(model):
    raw = sum(p.numel() for p in model.parameters())
    # Params4bit = packed 4-bit weights -> each stored element holds 2 real weights
    real = sum(p.numel() * (2 if p.__class__.__name__ == "Params4bit" else 1) for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return raw, real, trainable

raw, real, trainable = count_params(model)
before = {"total (numel, packed)": raw, "total (real)": real, "trainable": trainable}
print(f"numel() total (packed, misleading): {raw:,}")
print(f"Real total (4-bit corrected):       {real:,}")
print(f"Trainable right now:                {trainable:,}  ({100 * trainable / real:.2f}%)")
print("\n(anything 'trainable right now' is non-quantized weights still flagged trainable; get_peft_model freezes all of it except LoRA)")

### Parameter names and shapes
One or two names show how the Transformer is organised and where names like `q_proj` and `gate_proj` (our LoRA targets) come from. Note the quantized `q_proj` shape: it's packed, not the real 896×896.

In [ ]:
for name, param in model.named_parameters():
    print(f"{name:60s} {str(tuple(param.shape)):18s} {param.__class__.__name__}")
    if "layers.0.self_attn.q_proj.weight" in name:
        break

## 4. Apply LoRA / QLoRA
- `r`: LoRA rank. Higher rank = more adapter capacity and more trainable parameters.
- `lora_alpha`: scaling applied to the LoRA update (effective scale = `alpha / r` = 2 here).
- `lora_dropout = 0.0`: what Unsloth's fused kernels are optimised for.
- `target_modules`: every linear projection, attention **and** MLP.
- `bias="none"`: biases are not trained.
- `use_gradient_checkpointing="unsloth"`: recomputes activations instead of storing them, which saves VRAM.

For a linear layer `d_in → d_out`, LoRA learns **A: `r × d_in`** and **B: `d_out × r`** instead of a full `d_out × d_in` update:

**trainable ≈ `r × (d_in + d_out)`**. For `q_proj` (896→896, r=16) that's 28,672 instead of 802,816, about 28× fewer.

In [ ]:
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()      # PEFT's own count, already 4-bit corrected

### Verify what became trainable — AFTER QLoRA
Only `lora_A` / `lora_B` should show up. Check the formula on a real layer: `k_proj` is 896→128, so `lora_A` = 16×896 = 14,336 and `lora_B` = 128×16 = 2,048.

In [ ]:
trainable_params = [(name, p.numel()) for name, p in model.named_parameters() if p.requires_grad]
raw, real, trainable = count_params(model)

print(f"Real total:   {real:,}")
print(f"Trainable:    {trainable:,}  ({100 * trainable / real:.2f}%)")
print(f"Only LoRA params trainable: {all('lora_' in n for n, _ in trainable_params)}")

print("\nSample trainable parameters:")
for name, n in trainable_params[:10]:
    print(f"{name:80s} {n:,}")

### Total vs trainable: before and after QLoRA
The whole point of PEFT in one table: the model gets slightly **bigger** (the adapters are added), but the trainable part goes from "whatever wasn't frozen" to **only the LoRA matrices**.

In [ ]:
after = {"total (numel, packed)": raw, "total (real)": real, "trainable": trainable}
param_table = pd.DataFrame({"before QLoRA": before, "after QLoRA": after})
param_table.loc["trainable %"] = [100 * before["trainable"] / before["total (real)"],
                                  100 * after["trainable"] / after["total (real)"]]
param_table.style.format("{:,.0f}", subset=pd.IndexSlice[["total (numel, packed)", "total (real)", "trainable"], :]) \
                 .format("{:.2f}%", subset=pd.IndexSlice[["trainable %"], :])

### What one LoRA-wrapped layer looks like
One decoder layer instead of the whole 24-layer printout. Each projection is now `base_layer` (frozen, 4-bit) + `lora_A` → `lora_B` (trainable).

In [ ]:
print(model.base_model.model.model.layers[0])

## 5. Data
- `.rstrip()` makes every example end **exactly** on `<|im_end|>`. With a trailing `\n`, TRL can decide the text "doesn't end with EOS" and append a second one.
- Only the `text` column is kept. With a `messages` column present, TRL may treat the dataset as a chat dataset and re-apply the template itself (a second formatting path we don't want).
- **Smoke test** uses the 160 **longest** examples, so the VRAM it measures is the worst case.

In [ ]:
from typing import Any, cast

from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": str(DATA_DIR / "sft" / "train.jsonl"),
    "validation": str(DATA_DIR / "sft" / "val.jsonl"),
})
dataset = dataset.map(lambda x: {"text": x["text"].rstrip()})
dataset = cast(Any, dataset.remove_columns(["id", "messages"]))

print(dataset)

## 6. Minimal dataset sanity checks
Before spending GPU time, verify that:
1. the `text` field exists,
2. there are no empty examples,
3. the example really has the system → user → assistant structure, **with our system prompt** (not Qwen's default).

In [ ]:
for split in ["train", "validation"]:
    assert "text" in dataset[split].column_names
    empty = sum(not t.strip() for t in dataset[split]["text"])
    print(f"{split}: {len(dataset[split]):,} examples | empty: {empty}")
    assert empty == 0

sample_text = dataset["train"][0]["text"]
assert sample_text.startswith("<|im_start|>system\nYou are a function-calling assistant")
assert "You are Qwen" not in sample_text
assert sample_text.endswith("<|im_end|>")

print("\nSample training example:")
print(sample_text)

## 7. Token-length sanity check
This is the check that **caught the 512 bug**: 273 of 500 examples were over the limit. With `MAX_SEQ_LENGTH = 1536` from data prep, this should now say **0**. The data-prep percentiles are printed alongside so you can compare.

In [ ]:
audit = dataset["train"].select(range(min(500, len(dataset["train"]))))
lengths = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in audit["text"]]

print("Checked:", len(lengths), "training examples")
print("Max length: ", max(lengths))
print("Mean length:", round(sum(lengths) / len(lengths), 1))
print(f"Over MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH}):", sum(x > MAX_SEQ_LENGTH for x in lengths))
print("Data-prep percentiles (full train set):", sft_report["train_token_percentiles"])
assert max(lengths) <= MAX_SEQ_LENGTH

### Train/val sets for this run
Smoke test: the 160 **longest** training examples (worst-case VRAM) + 50 val. Real run: everything.

In [ ]:
train_ds, val_ds = dataset["train"], dataset["validation"]
full_train_chars = sum(len(t) for t in train_ds["text"])        # used for the time estimate

if SMOKE_TEST:
    char_lengths = [len(t) for t in train_ds["text"]]
    longest = sorted(range(len(char_lengths)), key=lambda i: char_lengths[i], reverse=True)
    train_ds = train_ds.select(longest[: SMOKE_STEPS * BATCH_SIZE * GRAD_ACCUM])
    val_ds = val_ds.select(range(50))

print(f"train: {len(train_ds):,} | val: {len(val_ds):,}")

## 8. SFT configuration + trainer
- **bf16** because the model was loaded in bf16 (that was the original `TypeError`).
- **`train_on_responses_only`**: loss only on the JSON answer. The system prompt and tool list are ~90% of each example (`median_share_of_tokens_trained` = 9% in the data report), so without masking most of the gradient would go into memorizing boilerplate.
- **`load_best_model_at_end`**: the HF version of the early-stopping-on-val-loss you wrote by hand for SmolLM2. `save_total_limit=2` keeps the best checkpoint plus the latest one.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

eval_every = 5 if SMOKE_TEST else EVAL_EVERY

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    max_steps=SMOKE_STEPS if SMOKE_TEST else -1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,                 # < 1 means a ratio of total steps
    weight_decay=0.01,
    optim="adamw_8bit",

    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),

    logging_steps=1 if SMOKE_TEST else 10,
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part=RESPONSE_MARKER,
)

## 9. Label-masking sanity check (before spending GPU hours)
If any of these fail, **stop**. They are the bugs that cost a whole training day:
1. **No example was dropped.** With 512 tokens, Unsloth dropped 22k.
2. **Only the answer is trained:** the unmasked tokens decode to exactly `answer + <|im_end|>`. That's the same check as in data prep, now run on what the trainer will *actually* use.

In [ ]:
assert len(trainer.train_dataset) == len(train_ds), \
    f"{len(train_ds) - len(trainer.train_dataset)} train examples dropped - is MAX_SEQ_LENGTH too small?"
assert len(trainer.eval_dataset) == len(val_ds), \
    f"{len(val_ds) - len(trainer.eval_dataset)} val examples dropped"

def trained_part(row):
    return tokenizer.decode([t for t, l in zip(row["input_ids"], row["labels"]) if l != -100])

for i in range(min(200, len(train_ds))):
    expected = train_ds[i]["text"].split(RESPONSE_MARKER)[-1]      # the JSON answer + <|im_end|>
    got = trained_part(trainer.train_dataset[i])
    assert got == expected, f"row {i}: trained on\n{got!r}\nexpected\n{expected!r}"

row = trainer.train_dataset[0]
n_trained = sum(l != -100 for l in row["labels"])
print(f"✅ nothing dropped ({len(trainer.train_dataset):,} train / {len(trainer.eval_dataset):,} val)")
print(f"✅ only the answer is trained, e.g. {n_trained} of {len(row['labels'])} tokens: {trained_part(row)!r}")

## 10. Train
Before the real run: **plug the laptop in and set Windows sleep to "Never"**. If it still crashes, re-run the notebook. It **resumes from the last checkpoint** (saved every 200 steps) instead of starting over.

The adapter is saved straight after training, so Part B can be re-run later without retraining.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR)) if OUTPUT_DIR.exists() else None
print("Resuming from", last_checkpoint if last_checkpoint else "scratch")

torch.cuda.reset_peak_memory_stats()
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
peak_train_vram = torch.cuda.max_memory_allocated() / 1024**3

model.save_pretrained(str(ADAPTER_DIR))         # best checkpoint (load_best_model_at_end)
tokenizer.save_pretrained(str(ADAPTER_DIR))
(RESULTS_DIR / "train_log.json").write_text(json.dumps(trainer.state.log_history, indent=2))

print(f"\nTrain loss:      {train_result.training_loss:.4f}")
print(f"Best eval loss:  {trainer.state.best_metric:.4f}  ({trainer.state.best_model_checkpoint})")
print(f"Runtime:         {train_result.metrics['train_runtime'] / 3600:.2f} h")
print(f"Peak VRAM:       {peak_train_vram:.2f} GB of 6")
print(f"Adapter saved -> {ADAPTER_DIR}")

if SMOKE_TEST:
    smoke_chars = sum(len(t) for t in train_ds["text"])
    est_hours = train_result.metrics["train_runtime"] / smoke_chars * full_train_chars * NUM_EPOCHS / 3600
    print(f"\n⏱  Full run estimate: ~{est_hours:.1f} h of training (upper bound: smoke used the longest examples)")
    print("   + evaluation: about 45 min on test_v1 (the baseline took ~0.95 s/question)")
    if peak_train_vram > 5.5:
        print("⚠️  Peak VRAM is close to 6 GB. Set BATCH_SIZE=1, GRAD_ACCUM=16 for the real run.")

### Loss curves
What you want to see: eval loss falls and then flattens. If it **rises** while train loss keeps falling, that's overfitting. `load_best_model_at_end` already kept the best checkpoint, so you're protected either way.

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(9, 3.5))
log.dropna(subset=["loss"]).plot(x="step", y="loss", ax=ax, label="train loss", alpha=0.6)
log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, label="eval loss", marker="o")
ax.set_title(f"{RUN_NAME}")
plt.tight_layout()
plt.show()

## 11. Post-training parameter summary
This answers one engineering question: **how big is the base model, and how much did we actually train?** The adapter folder is tiny compared with the model because the original weights never changed.

In [ ]:
raw, real, trainable = count_params(model)
adapter_mb = sum(f.stat().st_size for f in ADAPTER_DIR.glob("*.safetensors")) / 1024**2

print(f"Base + LoRA parameters (real): {real:,}")
print(f"Trained LoRA parameters:       {trainable:,}")
print(f"Trainable percentage:          {100 * trainable / real:.2f}%")
print(f"Adapter on disk:               {adapter_mb:.1f} MB  (vs ~1 GB for the full bf16 model)")

# Part B · Evaluate (identical to the baseline)
**Restarted the kernel, or the eval crashed?** Run sections 0 and 1, then continue from here. Part B only needs the saved adapter.

## 12. Build the eval model
The baseline was the **bf16** model loaded with plain `transformers`. To compare models and not loading methods, we do the same thing here: load the bf16 base, apply the adapter and **merge** it in. The merged model is saved, which is also the starting point for DPO later.

**Kernel check:** Unsloth patched `transformers` in this kernel during Part A. Before applying the adapter, the untouched base model must reproduce the saved baseline answers. If it does, this kernel generates the same way the baseline notebook did.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.evaluate import (breakdown, compare_runs, extract_json, failure_mix,      # noqa: E402
                          normalize_calls, query_template, score_generations, summarize, tag_features)
from src.generation import run_generation                                         # noqa: E402

# free the training model
trainer = model = None
gc.collect()
torch.cuda.empty_cache()

# frozen benchmark: refuse to run if it changed since the baseline
manifest = json.loads((EVAL_DIR / "manifest.json").read_text())
verify_hashes(EVAL_DIR, manifest["files"])
test_records = read_jsonl(EVAL_DIR / f"{BENCHMARK}.jsonl")
if SMOKE_TEST:
    test_records = test_records[:32]
records_by_id = {r["id"]: r for r in test_records}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype="auto", device_map="auto")
base_model.eval()

# --- kernel check: base model vs saved baseline answers ---
base_df_full = pd.read_csv(BASE_RESULTS_DIR / "scored.csv")
saved_base = base_df_full.set_index("id")["response"]
check_records = test_records[:16]
check_path = RESULTS_DIR / "kernel_check.jsonl"
check_path.unlink(missing_ok=True)
check_out = run_generation(base_model, tokenizer, check_records, build_messages, out_path=check_path,
                           run_config={"check": "base"}, batch_size=16, max_new_tokens=512)
same = sum(check_out[r["id"]]["response"] == saved_base[r["id"]] for r in check_records)
print(f"Kernel check: {same}/{len(check_records)} base answers identical to the baseline run")
if same < len(check_records) - 2:
    print("⚠️  Generation differs in this kernel. Restart the kernel, run sections 0 and 1, then Part B.")

# --- apply adapter and merge ---
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR)).merge_and_unload()
model.eval()
model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"Merged bf16 model saved -> {MERGED_DIR}")

## 13. Inference sanity check
Before the full benchmark, generate one answer and read it next to the gold answer. This goes through the same path the eval uses: **record → `build_messages` (our system prompt + tools + query) → chat template → model → JSON**.

(The old version passed only the user message, so Qwen silently inserted *"You are Qwen, created by Alibaba Cloud…"* as the system prompt. That's a prompt the model never trained on.)

In [ ]:
example = test_records[0]
messages = build_messages(example)
assert messages[0]["role"] == "system"            # our system prompt, not Qwen's default

inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.pad_token_id)

answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("QUERY:", example["query"])
print("GOLD :", json.dumps(example["answers"], ensure_ascii=False))
print("MODEL:", answer)
print("Parses as tool calls:", normalize_calls(extract_json(answer)) is not None)

## 14. Function-calling evaluation — generate on the frozen benchmark
Same benchmark, prompt version, greedy decoding and `max_new_tokens` as the baseline. Resumable: if this cell is interrupted, re-run it and it continues where it stopped.

In [ ]:
MAX_NEW_TOKENS = 512
RUN_CONFIG = {
    "model": f"{MODEL_NAME} + {RUN_NAME} (merged LoRA)", "benchmark": BENCHMARK,
    "benchmark_sha256": manifest["files"][f"{BENCHMARK}.jsonl"]["sha256"],
    "prompt_version": PROMPT_VERSION, "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy",
}

torch.cuda.reset_peak_memory_stats()
generations = run_generation(
    model, tokenizer, test_records, build_messages,
    out_path=RESULTS_DIR / "predictions.jsonl", run_config=RUN_CONFIG,
    batch_size=16, max_new_tokens=MAX_NEW_TOKENS,
)
peak_eval_vram = torch.cuda.max_memory_allocated() / 1024**3

## 15. Final comparison — base vs SFT
| Metric | Meaning |
|---|---|
| **`exact_match`** | every call fully correct, order ignored (**headline**) |
| `function_match` | right tools, ignoring argument values |
| `argument_f1` | partial credit per argument |
| `parse_valid` / `schema_valid` | valid JSON / real tools + required args |
| `invented_tool` / `invented_argument` | hallucinations (lower is better) |

- **Summary:** both runs scored on the same questions (in smoke mode the base numbers are recomputed on the same 32).
- **`compare_runs`:** `fixed` = questions SFT gets right that base got wrong, `broke` = the reverse. A McNemar p < 0.05 means the improvement is real and not noise.

In [ ]:
train_templates = {query_template(r["query"]) for r in read_jsonl(DATA_DIR / "splits" / "sft_train.jsonl")}
features_df = pd.DataFrame([{"id": r["id"], **tag_features(r, train_templates)} for r in test_records])

sft_df = score_generations(generations, records_by_id, features_df)
assert len(sft_df) == len(test_records), "some questions have no prediction"
base_df = base_df_full[base_df_full["id"].isin(sft_df["id"])]

sft_summary = summarize(sft_df)
base_summary = summarize(base_df)
summary_table = pd.DataFrame({
    "base": base_summary["value"],
    "sft": sft_summary["value"],
    "change": (sft_summary["value"] - base_summary["value"]).round(1),
    "sft 95% CI": sft_summary["ci_low"].astype(str) + " – " + sft_summary["ci_high"].astype(str),
})
display(summary_table)

comparison = compare_runs(base_df, sft_df)
comparison

### Where did it improve?
The row to watch is **`template_twin`**. Gains on `novel` questions mean real generalisation. Gains only on `seen_template` mean it mostly memorised patterns from training.

In [ ]:
FEATURES = ["call_pattern", "calls_bucket", "tools_bucket", "arg_complexity", "has_optional", "template_twin"]
for feature in FEATURES:
    side_by_side = breakdown(base_df, feature).rename(columns={"exact_match": "base"}).join(
        breakdown(sft_df, feature)["exact_match"].rename("sft"))
    side_by_side["change"] = (side_by_side["sft"] - side_by_side["base"]).round(1)
    print(f"\n=== {feature} ===")
    display(side_by_side)

display(pd.DataFrame({"base %": failure_mix(base_df), "sft %": failure_mix(sft_df)}).fillna(0))

In [ ]:
# eyeball one remaining failure of each type, so you can tell what DPO should target
for ftype, group in sft_df[sft_df["failure_type"] != "correct"].groupby("failure_type"):
    rec = records_by_id[group.iloc[0]["id"]]
    print(f"##### {ftype}  ({len(group)} questions)")
    print("QUERY :", rec["query"])
    print("GOLD  :", json.dumps(rec["answers"], ensure_ascii=False)[:300])
    print("MODEL :", str(group.iloc[0]["response"])[:300], "\n")

## 16. General-behaviour check (regression)
The same 20 prompts as the baseline. Expect plain chat to stay at ~0% tool-call JSON. **"No tool fits → `[]`" will probably not improve, and may drop:** xLAM has no examples where the right answer is `[]`. That's a known gap for the DPO stage, not a bug.

In [ ]:
general = read_jsonl(EVAL_DIR / "general_prompts_v1.jsonl")
chat_prompts = [g for g in general if g["kind"] == "chat"]
irrelevant_prompts = [g for g in general if g["kind"] == "irrelevant_tools"]

chat_out = run_generation(
    model, tokenizer, chat_prompts, lambda r: [{"role": "user", "content": r["query"]}],
    out_path=RESULTS_DIR / "general_chat.jsonl", run_config={**RUN_CONFIG, "set": "chat"},
    batch_size=16, max_new_tokens=200, stop_at_json=False,
)
irrelevant_out = run_generation(
    model, tokenizer, irrelevant_prompts, build_messages,
    out_path=RESULTS_DIR / "general_irrelevant.jsonl", run_config={**RUN_CONFIG, "set": "irrelevant"},
    batch_size=16, max_new_tokens=MAX_NEW_TOKENS,
)

chat_tool_rate = 100 * sum(bool(normalize_calls(extract_json(chat_out[g["id"]]["response"]))) for g in chat_prompts) / len(chat_prompts)
irrelevance_acc = 100 * sum(normalize_calls(extract_json(irrelevant_out[g["id"]]["response"])) == [] for g in irrelevant_prompts) / len(irrelevant_prompts)

base_general = json.loads((BASE_RESULTS_DIR / "metrics.json").read_text())["general_behaviour"]
print(f"Plain chat answered as tool-call JSON: base {base_general['chat_as_tool_call_pct']:.0f}% -> sft {chat_tool_rate:.0f}%  (lower is better)")
print(f"'No tool fits' correctly returned []:  base {base_general['irrelevance_returns_empty_pct']:.0f}% -> sft {irrelevance_acc:.0f}%  (higher is better)")

with pd.option_context("display.max_colwidth", 150):
    display(pd.DataFrame([{"query": g["query"], "sft response": chat_out[g["id"]]["response"][:150]} for g in chat_prompts]))

## 17. Save
Same files and `metrics.json` layout as the baseline, plus the training info. DPO will compare against `scored.csv` from here.

In [ ]:
sft_df.to_csv(RESULTS_DIR / "scored.csv", index=False)

metrics_report = {
    "run_name": RUN_NAME,
    "run_config": RUN_CONFIG,
    "n": len(sft_df),
    "summary": sft_summary.to_dict(orient="index"),
    "vs_base": comparison,
    "breakdowns": {f: breakdown(sft_df, f).to_dict(orient="index") for f in FEATURES},
    "failure_mix": failure_mix(sft_df).to_dict(),
    "generation": {
        "avg_gen_tokens": round(sft_df["gen_tokens"].mean(), 1),
        "hit_max_tokens_pct": round(100 * sft_df["hit_max_tokens"].mean(), 1),
        "latency_per_example_s": round(sft_df["latency_per_example"].mean(), 3),
        "peak_vram_gb": round(peak_eval_vram, 2),
    },
    "general_behaviour": {"chat_as_tool_call_pct": chat_tool_rate, "irrelevance_returns_empty_pct": irrelevance_acc},
    "training": {
        "model": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "epochs": NUM_EPOCHS, "effective_batch": BATCH_SIZE * GRAD_ACCUM, "learning_rate": LEARNING_RATE,
        "sft_data_files": sft_report["files"], "adapter_dir": str(ADAPTER_DIR), "merged_dir": str(MERGED_DIR),
    },
}
(RESULTS_DIR / "metrics.json").write_text(json.dumps(metrics_report, indent=2, default=str))
print(f"Saved to {RESULTS_DIR}")
if SMOKE_TEST:
    print("\n✅ Smoke test passed end-to-end. Set SMOKE_TEST = False and Run All for the real run.")